# 500m 체감온도 격자 → 100m 격자 공간 조인
## Projected CRS 기반 정밀 공간 매핑 파이프라인 (평균 체감온도)

---

### 목표
기상청 API의 **500m 해상도 평균 체감온도(ta_chi)** 데이터를  
**100m 해상도 분석 격자**에 공간적으로 정확하게 매핑한다.

> **입력 데이터**: 2025년 6~8월(92일) 체감온도를 **위치별로 평균**낸  
> `ta_chi_seoul_mean_2025.csv` (약 2,304개 격자점 × 평균 ta_chi)

### 방법론

| 단계 | 내용 |
|------|------|
| **좌표계** | WGS84(EPSG:4326) → Korea 2000 / Unified CS(**EPSG:5179**) 투영 변환 |
| **폴리곤 생성** | 500m 중심점에서 **정확히 ±250m** 오프셋으로 정사각형 폴리곤 생성 (미터 단위) |
| **공간 조인** | 100m 격자 중심점이 어떤 500m 폴리곤 **내부(within)** 에 포함되는지 판정 |

### 왜 투영 좌표계(EPSG:5179)인가?

지리 좌표계(EPSG:4326)에서는 1°의 실제 거리가 위도에 따라 달라진다:  
- 위도 1° ≈ 111km (일정)  
- **경도 1° ≈ 111km × cos(φ)** (위도에 의존)  

따라서 WGS84 좌표에서 ±0.00225° 같은 근사치로 격자를 생성하면  
**격자 크기가 위치마다 달라지는 체계적 오차**가 발생한다.  

**EPSG:5179**(Korea 2000 / Unified CS)는:  
- 대한민국 전역에 최적화된 **횡축 메르카토르(TM) 투영**  
- 좌표 단위가 **미터(metre)** → `box(x-250, y-250, x+250, y+250)` = 정확히 500m × 500m  
- 서울 지역에서 거리 왜곡 < 0.01% → 학술 분석에 적합  

### 왜 공간 조인이 KD-Tree 최근접 이웃보다 정확한가?

| | KD-Tree 최근접 이웃 | 투영 좌표계 + 공간 조인 |
|---|---|---|
| 격자 경계 | 근사적 (cos 보정) | **정확** (미터 기반 폴리곤) |
| 경계 판정 | 거리 기반 → 경계 모호 | **within** 연산 → 명확한 포함 관계 |
| 재현성 | 스케일링 파라미터 의존 | **CRS 코드만으로 재현** |
| 학술 적합성 | 방법론 기술 어려움 | **표준 GIS 워크플로우** |

---
## 1. 라이브러리 및 환경 설정

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely

print(f"pandas     {pd.__version__}")
print(f"numpy      {np.__version__}")
print(f"geopandas  {gpd.__version__}")
print(f"shapely    {shapely.__version__}")

# Shapely 2.0+ 벡터화 연산 필요
assert int(shapely.__version__.split('.')[0]) >= 2, \
    f"Shapely 2.0 이상이 필요합니다. 현재: {shapely.__version__}"

pandas     2.2.3
numpy      2.1.3
geopandas  1.1.2
shapely    2.1.2


---
## 2. 데이터 로드

In [3]:
# ── 500m 평균 체감온도 데이터 (격자점 × 평균 ta_chi) ──
df_tachi = pd.read_csv("../data_processed/processed/ta_chi/ta_chi_seoul_mean_2025.csv")

print(f"평균 체감온도 데이터: {df_tachi.shape[0]:,}행 × {df_tachi.shape[1]}열")
print(f"  컬럼: {df_tachi.columns.tolist()}")
print(f"  고유 격자점: {df_tachi.groupby(['Latitude','Longitude']).ngroups:,}개")
print(f"  ta_chi 범위: {df_tachi['ta_chi'].min():.2f} ~ {df_tachi['ta_chi'].max():.2f}")
df_tachi.head(3)

평균 체감온도 데이터: 2,304행 × 3열
  컬럼: ['Latitude', 'Longitude', 'ta_chi']
  고유 격자점: 2,304개
  ta_chi 범위: 27.07 ~ 32.65


,Latitude,Longitude,ta_chi
0,37.430523,127.065697,29.74
1,37.430584,127.059906,29.35
2,37.430645,127.054115,28.98


In [4]:
# ── 100m 분석 격자 데이터 ──
df_grid = pd.read_csv("../data_processed/processed/grid/seoul_grid_100m.csv")

print(f"100m 격자 데이터: {df_grid.shape[0]:,}행 × {df_grid.shape[1]}열")
print(f"  컬럼: {df_grid.columns.tolist()}")
df_grid.head(3)

100m 격자 데이터: 92,398행 × 8열
  컬럼: ['system:index', 'min_lon', 'max_lat', 'center_lat', 'max_lon', 'center_lon', 'min_lat', 'cell_id']


,system:index,min_lon,max_lat,center_lat,max_lon,center_lon,min_lat,cell_id
0,+141141+45163,126.789118,37.552496,37.552140,126.790016,126.789567,37.551784,1411415045163
1,+141141+45164,126.789118,37.553208,37.552852,126.790016,126.789567,37.552496,1411415045164
2,+141141+45165,126.789118,37.553921,37.553564,126.790016,126.789567,37.553208,1411415045165


In [5]:
# ── 500m 격자의 고유 좌표 추출 및 ID 부여 ──
# 평균 데이터이므로 각 행이 고유 격자점이지만,
# 혹시 모를 좌표 중복을 처리하고 grid_500m_id를 부여한다.
coords_500m = (
    df_tachi[['Latitude', 'Longitude']]
    .drop_duplicates()
    .reset_index(drop=True)
)
coords_500m['grid_500m_id'] = np.arange(len(coords_500m))

# 원본 ta_chi 데이터에 grid_500m_id 부여
df_tachi = df_tachi.merge(coords_500m, on=['Latitude', 'Longitude'], how='left')

print(f"500m 고유 격자점: {len(coords_500m):,}개")
print(f"grid_500m_id 결측: {df_tachi['grid_500m_id'].isna().sum()}개")

500m 고유 격자점: 2,304개
grid_500m_id 결측: 0개


---
## 3. GeoDataFrame 생성 (WGS84, EPSG:4326)

원본 좌표(위경도)를 GeoPandas의 Point 객체로 변환한다.  
이 단계에서는 아직 지리 좌표계(EPSG:4326)를 사용한다.

In [6]:
# ── 500m 격자 중심점 → GeoDataFrame ──
# gpd.points_from_xy(경도, 위도) — x=경도, y=위도 순서
gdf_500m = gpd.GeoDataFrame(
    coords_500m,
    geometry=gpd.points_from_xy(coords_500m['Longitude'], coords_500m['Latitude']),
    crs='EPSG:4326'
)

# ── 100m 격자 중심점 → GeoDataFrame ──
gdf_100m = gpd.GeoDataFrame(
    df_grid,
    geometry=gpd.points_from_xy(df_grid['center_lon'], df_grid['center_lat']),
    crs='EPSG:4326'
)

print(f"500m GeoDataFrame: {len(gdf_500m):,}행, CRS = {gdf_500m.crs}")
print(f"100m GeoDataFrame: {len(gdf_100m):,}행, CRS = {gdf_100m.crs}")

500m GeoDataFrame: 2,304행, CRS = EPSG:4326
100m GeoDataFrame: 92,398행, CRS = EPSG:4326


---
## 4. 투영 좌표계 변환 (EPSG:5179)

**EPSG:5179** (Korea 2000 / Unified CS)  
- 투영법: Transverse Mercator  
- 중앙 경선: 127.5°E (한반도 중앙)  
- 좌표 단위: **미터(metre)**  
- 적용 범위: 대한민국 전역  

투영 변환 후 좌표 단위가 미터임을 검증한다.

In [7]:
TARGET_CRS = 'EPSG:5179'

# ── 투영 변환 ──
gdf_500m_proj = gdf_500m.to_crs(TARGET_CRS)
gdf_100m_proj = gdf_100m.to_crs(TARGET_CRS)

# ── CRS 검증 ──
# 1) 좌표 단위가 미터(metre)인지 확인
unit = gdf_500m_proj.crs.axis_info[0].unit_name
assert unit == 'metre', f"CRS 좌표 단위 오류: '{unit}' (기대: 'metre')"

# 2) 두 데이터셋의 CRS 일치 확인
assert gdf_500m_proj.crs == gdf_100m_proj.crs, \
    "CRS 불일치: 500m과 100m 데이터의 좌표계가 다릅니다"

print(f"투영 CRS   : {gdf_500m_proj.crs}")
print(f"좌표 단위  : {unit}")
print(f"CRS 일치   : ✓")
print(f"\n500m 투영 좌표 범위:")
print(f"  X: {gdf_500m_proj.geometry.x.min():,.0f} ~ {gdf_500m_proj.geometry.x.max():,.0f} m")
print(f"  Y: {gdf_500m_proj.geometry.y.min():,.0f} ~ {gdf_500m_proj.geometry.y.max():,.0f} m")

투영 CRS   : EPSG:5179
좌표 단위  : metre
CRS 일치   : ✓

500m 투영 좌표 범위:
  X: 935,669 ~ 971,589 m
  Y: 1,936,907 ~ 1,966,714 m


---
## 5. 500m × 500m 정사각형 폴리곤 생성

투영 좌표계(미터 단위)에서 각 500m 격자 중심점으로부터  
**정확히 ±250m** 오프셋을 적용하여 정사각형 폴리곤을 생성한다.

```
box(x - 250, y - 250, x + 250, y + 250)
```

- 투영 좌표의 단위가 미터이므로, 이 연산은 **정확히 500m × 500m** 정사각형을 생성한다.
- `shapely.box()`의 벡터화 연산(Shapely 2.0+)으로 per-row apply 없이 일괄 생성한다.

In [8]:
HALF_GRID = 250  # 500m 격자의 절반 = 250m

# ── 투영 좌표 추출 (numpy array) ──
x = gdf_500m_proj.geometry.x.values
y = gdf_500m_proj.geometry.y.values

# ── 벡터화 폴리곤 생성 (Shapely 2.0+) ──
# shapely.box()는 numpy array를 입력받아 한 번에 모든 폴리곤을 생성한다.
polygons = shapely.box(
    x - HALF_GRID,  # xmin
    y - HALF_GRID,  # ymin
    x + HALF_GRID,  # xmax
    y + HALF_GRID   # ymax
)

# ── 폴리곤 GeoDataFrame 구성 ──
gdf_poly = gpd.GeoDataFrame(
    {
        'grid_500m_id': coords_500m['grid_500m_id'].values,
        'center_x_5179': x,   # 500m 중심점 투영 X좌표 (진단용)
        'center_y_5179': y,   # 500m 중심점 투영 Y좌표 (진단용)
    },
    geometry=polygons,
    crs=TARGET_CRS
)

print(f"500m 폴리곤 수: {len(gdf_poly):,}개")

# ── 폴리곤 크기 검증 ──
sample = gdf_poly.geometry.iloc[0]
width  = sample.bounds[2] - sample.bounds[0]
height = sample.bounds[3] - sample.bounds[1]
area   = sample.area

print(f"\n폴리곤 크기 검증 (첫 번째 격자):")
print(f"  가로: {width:.1f} m  (기대: 500.0 m)")
print(f"  세로: {height:.1f} m  (기대: 500.0 m)")
print(f"  넓이: {area:,.0f} m²  (기대: 250,000 m²)")

assert abs(width - 500) < 0.01, f"가로 크기 오류: {width}"
assert abs(height - 500) < 0.01, f"세로 크기 오류: {height}"

500m 폴리곤 수: 2,304개

폴리곤 크기 검증 (첫 번째 격자):
  가로: 500.0 m  (기대: 500.0 m)
  세로: 500.0 m  (기대: 500.0 m)
  넓이: 250,000 m²  (기대: 250,000 m²)


In [ ]:
# ── (선택) 500m 폴리곤 GeoJSON 저장 ──
# 시각화/검증 목적으로 WGS84로 변환하여 저장
# output_geojson = "../all_data/ta_chi_500m_polygons.geojson"
# gdf_poly.to_crs('EPSG:4326').to_file(output_geojson, driver='GeoJSON')
# print(f"500m 폴리곤 GeoJSON 저장: {output_geojson}")

---
## 6. 공간 조인 (Spatial Join)

100m 격자 중심점(Point)이 어떤 500m 폴리곤(Polygon) **내부**에 포함되는지 판정한다.

- `predicate='within'`: 점이 폴리곤 내부에 있을 때만 매칭  
- GeoPandas는 내부적으로 **R-tree 공간 인덱스**를 사용하여 효율적으로 조인을 수행한다.  
- 이 조인은 고유 격자점(~2,300개)에 대해 **한 번만** 수행하며,  
  평균 ta_chi 값은 이후 테이블 조인으로 매핑한다.

In [9]:
# ── 공간 조인: 100m 점(Point) ∈ 500m 폴리곤(Polygon) ──
joined = gpd.sjoin(
    gdf_100m_proj,
    gdf_poly[['grid_500m_id', 'center_x_5179', 'center_y_5179', 'geometry']],
    how='left',
    predicate='within'
)

# ── 중복 매칭 검사 ──
# 정규 격자에서는 한 점이 두 폴리곤에 동시에 포함될 수 없다.
# 경계에 정확히 위치하는 극히 드문 경우를 대비해 중복을 검사한다.
n_dup = joined.index.duplicated().sum()
if n_dup > 0:
    print(f"⚠ 중복 매칭 {n_dup}건 발견 → 첫 번째 매칭만 유지")
    joined = joined[~joined.index.duplicated(keep='first')]
else:
    print(f"중복 매칭: 없음 ✓")

print(f"공간 조인 결과: {len(joined):,}행")

중복 매칭: 없음 ✓
공간 조인 결과: 92,398행


In [10]:
# ── 미매칭 100m 셀 → 최근접 500m 격자로 보완 매핑 (Nearest Neighbor Fallback) ──
# within 조인에서 매칭되지 않은 셀을 가장 가까운 500m 격자 중심점으로 매핑한다.
# 원인: 기상청 격자 커버리지 경계 밖, 행정경계 불일치, 하천/해안 영역 등

unmatched_mask = joined['grid_500m_id'].isna()
n_unmatched = unmatched_mask.sum()

# 매칭 방법 추적 컬럼 추가
joined['match_method'] = 'spatial_join'
joined.loc[unmatched_mask, 'match_method'] = 'nearest_neighbor'

if n_unmatched > 0:
    print(f"미매칭 100m 셀: {n_unmatched:,}개 → 최근접 이웃(nearest neighbor)으로 보완 매핑\n")

    # 미매칭 셀의 투영 좌표 GeoDataFrame 추출
    unmatched_idx = joined.loc[unmatched_mask].index
    unmatched_gdf = gdf_100m_proj.loc[unmatched_idx].copy()

    # 500m 격자 중심점 참조 테이블 (투영 좌표 포함)
    gdf_500m_ref = gdf_500m_proj[['grid_500m_id', 'geometry']].copy()
    gdf_500m_ref['center_x_5179'] = gdf_500m_ref.geometry.x
    gdf_500m_ref['center_y_5179'] = gdf_500m_ref.geometry.y

    # sjoin_nearest: 미매칭 점 → 가장 가까운 500m 격자 중심점 (투영 좌표 기반, 미터 단위)
    nearest = gpd.sjoin_nearest(
        unmatched_gdf[['cell_id', 'geometry']],
        gdf_500m_ref,
        how='left',
        distance_col='nearest_dist_m'
    )

    # 동일 거리에 복수 격자가 있을 경우 첫 번째만 유지
    nearest = nearest[~nearest.index.duplicated(keep='first')]

    # joined 테이블에 최근접 결과 반영
    joined.loc[unmatched_mask, 'grid_500m_id'] = nearest['grid_500m_id'].values
    joined.loc[unmatched_mask, 'center_x_5179'] = nearest['center_x_5179'].values
    joined.loc[unmatched_mask, 'center_y_5179'] = nearest['center_y_5179'].values

    # 검증 출력
    n_still_na = joined['grid_500m_id'].isna().sum()
    print("=" * 55)
    print("  최근접 이웃 보완 매핑 결과")
    print("=" * 55)
    print(f"  보완 매핑 대상   : {n_unmatched:>8,}개")
    print(f"  보완 매핑 완료   : {n_unmatched - n_still_na:>8,}개")
    print(f"  여전히 미매칭    : {n_still_na:>8}개")
    print("=" * 55)
    print(f"\n최근접 매핑 거리 통계 (투영 좌표 기반, 미터):")
    print(f"  평균   : {nearest['nearest_dist_m'].mean():>8.1f} m")
    print(f"  중앙값 : {nearest['nearest_dist_m'].median():>8.1f} m")
    print(f"  최소   : {nearest['nearest_dist_m'].min():>8.1f} m")
    print(f"  최대   : {nearest['nearest_dist_m'].max():>8.1f} m")
    print(f"  표준편차: {nearest['nearest_dist_m'].std():>8.1f} m")
else:
    print("미매칭 없음 — 모든 100m 셀이 500m 폴리곤에 포함됨 ✓")

미매칭 100m 셀: 15,018개 → 최근접 이웃(nearest neighbor)으로 보완 매핑

  최근접 이웃 보완 매핑 결과
  보완 매핑 대상   :   15,018개
  보완 매핑 완료   :   15,018개
  여전히 미매칭    :        0개

최근접 매핑 거리 통계 (투영 좌표 기반, 미터):
  평균   :    773.1 m
  중앙값 :    606.1 m
  최소   :    250.0 m
  최대   :   2622.0 m
  표준편차:    522.9 m


---
## 7. 검증

In [11]:
# ── 7-1. 최종 매칭 커버리지 검증 ──
n_total     = len(joined)
n_matched   = joined['grid_500m_id'].notna().sum()
n_unmatched = n_total - n_matched
n_spatial   = (joined['match_method'] == 'spatial_join').sum()
n_nearest   = (joined['match_method'] == 'nearest_neighbor').sum()

print("=" * 55)
print("  최종 매칭 검증 결과 (공간 조인 + 최근접 이웃)")
print("=" * 55)
print(f"  전체 100m 격자  : {n_total:>8,}")
print(f"  매칭 성공        : {n_matched:>8,}  ({n_matched / n_total * 100:.2f}%)")
print(f"    ├ 공간 조인    : {n_spatial:>8,}  ({n_spatial / n_total * 100:.2f}%)")
print(f"    └ 최근접 이웃  : {n_nearest:>8,}  ({n_nearest / n_total * 100:.2f}%)")
print(f"  매칭 실패        : {n_unmatched:>8,}  ({n_unmatched / n_total * 100:.2f}%)")
print("=" * 55)

if n_unmatched > 0:
    print(f"\n⚠ 여전히 미매칭인 셀: {n_unmatched:,}개")
else:
    print(f"\n✓ 모든 100m 격자에 체감온도 데이터가 매핑되었습니다.")

  최종 매칭 검증 결과 (공간 조인 + 최근접 이웃)
  전체 100m 격자  :   92,398
  매칭 성공        :   92,398  (100.00%)
    ├ 공간 조인    :   77,380  (83.75%)
    └ 최근접 이웃  :   15,018  (16.25%)
  매칭 실패        :        0  (0.00%)

✓ 모든 100m 격자에 체감온도 데이터가 매핑되었습니다.


In [12]:
# ── 7-2. 매칭 거리 진단 ──
# 100m 중심점 → 매칭된 500m 격자 중심점 간 유클리드 거리 (투영 좌표, 미터)
joined['dist_to_center_m'] = np.sqrt(
    (joined.geometry.x - joined['center_x_5179']) ** 2 +
    (joined.geometry.y - joined['center_y_5179']) ** 2
)

DIAG_MAX = HALF_GRID * np.sqrt(2)  # 이론적 최대 거리 ≈ 353.6m

# ── (A) 공간 조인(within) 매칭 거리 ──
spatial_mask = joined['match_method'] == 'spatial_join'
dist_spatial = joined.loc[spatial_mask, 'dist_to_center_m']

print("[ 공간 조인(within) 매칭 거리 ] (투영 좌표 기반, 미터)")
print(f"  대상   : {spatial_mask.sum():,}개")
print(f"  평균   : {dist_spatial.mean():.1f} m")
print(f"  중앙값 : {dist_spatial.median():.1f} m")
print(f"  최대   : {dist_spatial.max():.1f} m")
print(f"  표준편차: {dist_spatial.std():.1f} m")
print(f"  이론적 최대 (대각선): {DIAG_MAX:.1f} m")

n_over = (dist_spatial > DIAG_MAX).sum()
print(f"  이론 최대 초과: {n_over}건")
assert n_over == 0, f"공간 조인에서 이론적 최대 거리를 초과하는 매칭이 {n_over}건 존재합니다"

# ── (B) 최근접 이웃(nearest) 매칭 거리 ──
nearest_mask = joined['match_method'] == 'nearest_neighbor'
if nearest_mask.sum() > 0:
    dist_nearest = joined.loc[nearest_mask, 'dist_to_center_m']
    print(f"\n[ 최근접 이웃(nearest) 매칭 거리 ] (투영 좌표 기반, 미터)")
    print(f"  대상   : {nearest_mask.sum():,}개")
    print(f"  평균   : {dist_nearest.mean():.1f} m")
    print(f"  중앙값 : {dist_nearest.median():.1f} m")
    print(f"  최소   : {dist_nearest.min():.1f} m")
    print(f"  최대   : {dist_nearest.max():.1f} m")
    print(f"  표준편차: {dist_nearest.std():.1f} m")

# ── (C) 전체 통계 ──
dist_all = joined['dist_to_center_m']
print(f"\n[ 전체 매칭 거리 통계 ]")
print(f"  대상   : {len(dist_all):,}개")
print(f"  평균   : {dist_all.mean():.1f} m")
print(f"  중앙값 : {dist_all.median():.1f} m")
print(f"  최대   : {dist_all.max():.1f} m")

[ 공간 조인(within) 매칭 거리 ] (투영 좌표 기반, 미터)
  대상   : 77,380개
  평균   : 191.4 m
  중앙값 : 199.6 m
  최대   : 352.8 m
  표준편차: 71.2 m
  이론적 최대 (대각선): 353.6 m
  이론 최대 초과: 0건

[ 최근접 이웃(nearest) 매칭 거리 ] (투영 좌표 기반, 미터)
  대상   : 15,018개
  평균   : 773.1 m
  중앙값 : 606.1 m
  최소   : 250.0 m
  최대   : 2622.0 m
  표준편차: 522.9 m

[ 전체 매칭 거리 통계 ]
  대상   : 92,398개
  평균   : 285.9 m
  중앙값 : 218.0 m
  최대   : 2622.0 m


---
## 8. 최종 데이터 구성

공간 조인으로 얻은 **100m cell → 500m grid 매핑 테이블**을 이용하여  
평균 체감온도를 100m 격자에 매핑한다.

```
df_mapping (cell_id → grid_500m_id)  ×  df_tachi (grid_500m_id → 평균 ta_chi)
→ 100m 격자별 평균 ta_chi
```

In [13]:
# ── 8-1. 매핑 테이블 구성 ──
df_mapping = (
    joined[['cell_id', 'grid_500m_id', 'dist_to_center_m', 'match_method']]
    .dropna(subset=['grid_500m_id'])
    .copy()
)
df_mapping['grid_500m_id'] = df_mapping['grid_500m_id'].astype(int)

print(f"매핑 테이블: {len(df_mapping):,}행 (전체 100m 셀)")
print(f"  공간 조인  : {(df_mapping['match_method'] == 'spatial_join').sum():,}행")
print(f"  최근접 이웃: {(df_mapping['match_method'] == 'nearest_neighbor').sum():,}행")

# ── 8-2. 평균 체감온도 매핑 ──
# 평균 데이터이므로 DateTime 없이 grid_500m_id 기준으로 바로 매핑
df_final = df_mapping.merge(
    df_tachi[['grid_500m_id', 'ta_chi']],
    on='grid_500m_id',
    how='inner'
)

# ── 8-3. 격자 메타데이터 결합 ──
df_final = df_final.merge(
    df_grid[['cell_id', 'center_lat', 'center_lon']],
    on='cell_id',
    how='left'
)

# ── 8-4. 컬럼 정리 ──
df_final = df_final[[
    'cell_id',
    'center_lat', 'center_lon',
    'ta_chi', 'dist_to_center_m', 'match_method'
]].rename(columns={
    'dist_to_center_m': 'distance_to_polygon_center'
})

print(f"\n최종 데이터: {df_final.shape[0]:,}행 × {df_final.shape[1]}열")
print(f"  격자 수: {df_final['cell_id'].nunique():,}")
print(f"  ta_chi 결측: {df_final['ta_chi'].isna().sum()}")
print(f"  ta_chi 범위: {df_final['ta_chi'].min():.2f} ~ {df_final['ta_chi'].max():.2f}")
print(f"  매칭 방법 분포:")
print(f"    spatial_join     : {(df_final['match_method'] == 'spatial_join').sum():,}")
print(f"    nearest_neighbor : {(df_final['match_method'] == 'nearest_neighbor').sum():,}")
df_final.head(10)

매핑 테이블: 92,398행 (전체 100m 셀)
  공간 조인  : 77,380행
  최근접 이웃: 15,018행

최종 데이터: 92,398행 × 6열
  격자 수: 92,398
  ta_chi 결측: 0
  ta_chi 범위: 27.49 ~ 32.65
  매칭 방법 분포:
    spatial_join     : 77,380
    nearest_neighbor : 15,018


,cell_id,center_lat,center_lon,ta_chi,distance_to_polygon_center,match_method
0,1411415045163,37.552140,126.789567,30.55,120.558717,spatial_join
1,1411415045164,37.552852,126.789567,30.55,49.575922,spatial_join
2,1411415045165,37.553564,126.789567,30.55,53.553898,spatial_join
3,1411425045158,37.548579,126.790465,30.51,117.144457,spatial_join
4,1411425045159,37.549291,126.790465,30.51,142.177738,spatial_join
5,1411425045160,37.550004,126.790465,30.51,197.974975,spatial_join
6,1411425045161,37.550716,126.790465,30.51,265.821751,spatial_join
7,1411425045162,37.551428,126.790465,30.55,225.049272,spatial_join
8,1411425045163,37.552140,126.790465,30.55,161.502861,spatial_join
9,1411425045164,37.552852,126.790465,30.55,118.348751,spatial_join


In [14]:
# ── 8-5. 최종 데이터 요약 ──
print(f"전체 행 수: {len(df_final):,}")
print(f"ta_chi 범위: {df_final['ta_chi'].min():.2f} ~ {df_final['ta_chi'].max():.2f}")
print(f"ta_chi 평균: {df_final['ta_chi'].mean():.2f}")
print(f"ta_chi 표준편차: {df_final['ta_chi'].std():.2f}")
print(f"\n컬럼: {df_final.columns.tolist()}")
df_final.describe()

전체 행 수: 92,398
ta_chi 범위: 27.49 ~ 32.65
ta_chi 평균: 30.19
ta_chi 표준편차: 0.54

컬럼: ['cell_id', 'center_lat', 'center_lon', 'ta_chi', 'distance_to_polygon_center', 'match_method']


,cell_id,center_lat,center_lon,ta_chi,distance_to_polygon_center
count,9.239800e+04,92398.000000,92398.000000,92398.000000,92398.000000
mean,1.413823e+12,37.538095,127.005892,30.185219,285.929991
std,9.913904e+08,0.060124,0.089058,0.543691,307.811720
min,1.411415e+12,37.425264,126.789567,27.490000,0.702849
25%,1.413085e+12,37.490154,126.939585,29.990000,154.227455
50%,1.413915e+12,37.534333,127.014145,30.300000,217.975178
75%,1.414605e+12,37.579198,127.076129,30.420000,271.074094
max,1.415945e+12,37.683778,127.196503,32.650000,2621.965646


---
## 9. 저장

In [15]:
# ── 9-0. 저장 전 cell_id 무결성 검증 ──
# 원본 100m 격자(df_grid)의 cell_id와 최종 데이터(df_final)의 cell_id가
# 정확히 일치하는지 확인한다.

grid_ids  = set(df_grid['cell_id'])
final_ids = set(df_final['cell_id'])

# 1) 개수 비교
print("=" * 55)
print("  cell_id 무결성 검증 (df_grid vs df_final)")
print("=" * 55)
print(f"  원본 격자(df_grid)  : {len(grid_ids):>8,}개")
print(f"  최종 데이터(df_final): {len(final_ids):>8,}개")

# 2) 집합 비교
missing_in_final = grid_ids - final_ids   # 원본에는 있지만 최종에 없는 cell_id
extra_in_final   = final_ids - grid_ids   # 최종에만 있는 cell_id (있으면 안 됨)

print(f"\n  원본에만 존재 (누락) : {len(missing_in_final):>8,}개")
print(f"  최종에만 존재 (초과) : {len(extra_in_final):>8,}개")

# 3) 중복 검사
dup_in_final = df_final['cell_id'].duplicated().sum()
print(f"  최종 데이터 중복     : {dup_in_final:>8,}개")

# 4) 완전 일치 판정
is_perfect = (grid_ids == final_ids) and (dup_in_final == 0)
print("=" * 55)

if is_perfect:
    print("✓ cell_id 완전 일치 — 원본 격자와 최종 데이터의 셀이 1:1로 매핑됨")
else:
    if len(missing_in_final) > 0:
        print(f"\n⚠ 누락된 cell_id 샘플 (최대 5개): {sorted(missing_in_final)[:5]}")
    if len(extra_in_final) > 0:
        print(f"\n⚠ 초과된 cell_id 샘플 (최대 5개): {sorted(extra_in_final)[:5]}")
    if dup_in_final > 0:
        print(f"\n⚠ 중복된 cell_id 수: {dup_in_final}개")
    assert False, (
        f"cell_id 불일치! 누락 {len(missing_in_final)}개, "
        f"초과 {len(extra_in_final)}개, 중복 {dup_in_final}개"
    )

  cell_id 무결성 검증 (df_grid vs df_final)
  원본 격자(df_grid)  :   92,398개
  최종 데이터(df_final):   92,398개

  원본에만 존재 (누락) :        0개
  최종에만 존재 (초과) :        0개
  최종 데이터 중복     :        0개
✓ cell_id 완전 일치 — 원본 격자와 최종 데이터의 셀이 1:1로 매핑됨


In [16]:
# ── 매핑 테이블 저장 (재사용 가능) ──
mapping_path = "../data_processed/processed/ta_chi/grid_100m_500m_spatial_mapping.csv"
df_mapping.to_csv(mapping_path, index=False)
print(f"매핑 테이블 저장: {mapping_path}")

# ── 최종 데이터 저장 ──
output_path = "../data_processed/processed/ta_chi/tachi_with_grid.csv"
df_final.to_csv(output_path, index=False)
print(f"최종 데이터 저장: {output_path}")
print(f"  행 수: {len(df_final):,}")
print(f"  파일 크기 (예상): ~{df_final.memory_usage(deep=True).sum() / 1024 / 1024:.0f} MB")

매핑 테이블 저장: ../data_processed/processed/ta_chi/grid_100m_500m_spatial_mapping.csv
최종 데이터 저장: ../data_processed/processed/ta_chi/tachi_with_grid.csv
  행 수: 92,398
  파일 크기 (예상): ~9 MB
